# Day 037 | Linear Regression with Scikit-Learn

## 100 Days of ML for Geotechnical Engineering
**Phase:** Classical Machine Learning

**Author:** Ripon Chandra Malo  

---

## What We Will Learn Today

| Concept | What It Means | Geotechnical Example |
|---------|--------------|---------------------|
| `LinearRegression` | sklearn's linear model class | Predict UCS from soil properties |
| Train/test split | Separate data for learning vs evaluation | 80% train, 20% test |
| `model.fit()` | Train the model on training data | Learn weight-feature relationships |
| `model.predict()` | Make predictions on new data | Estimate UCS for untested samples |
| R-squared (R²) | Fraction of variance explained | How well features predict UCS |
| Coefficients | Learned weights for each feature | Which soil property matters most |
| Residual analysis | Check model assumptions | Are errors randomly distributed? |

---

## Our Geotechnical Problem

**Predict Unconfined Compressive Strength (UCS) from multiple soil properties.**

UCS is a critical parameter for foundation design, slope stability, and tunnel engineering. Lab testing is expensive and time-consuming. Can we predict UCS from simpler measurements?

Features available:
- `depth` (m) — sample depth
- `N_value` — SPT blow count
- `water_content` (%) — gravimetric water content
- `void_ratio` — ratio of void volume to solid volume
- `unit_weight` (kN/m³) — bulk unit weight

Target: `UCS` (kPa) — unconfined compressive strength

Yesterday we built linear regression from scratch. Today we use **scikit-learn** to do the same thing in a few lines — plus handle multiple features, train/test splitting, and proper evaluation.

---

## Part 1: Load and Explore the Data

In [ ]:
# ─────────────────────────────────────────────────────
# IMPORTS
# ─────────────────────────────────────────────────────

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import train_test_split
from sklearn.metrics import r2_score, mean_squared_error, mean_absolute_error

np.random.seed(37)

print("Libraries loaded successfully.")

In [ ]:
# ─────────────────────────────────────────────────────
# LOAD THE UCS DATASET
# ─────────────────────────────────────────────────────

df = pd.read_csv("ucs_data.csv")

print(f"Dataset shape: {df.shape[0]} samples × {df.shape[1]} columns")
print()
print("First 10 rows:")
df.head(10)

In [ ]:
# ─────────────────────────────────────────────────────
# BASIC STATISTICS
# ─────────────────────────────────────────────────────

print("Summary Statistics:")
print("═" * 70)
print(df.describe().round(2).to_string())
print("═" * 70)
print()
print("Missing values:")
print(df.isnull().sum().to_string())
print()
print("Data types:")
print(df.dtypes.to_string())

In [ ]:
# ─────────────────────────────────────────────────────
# VISUALIZE FEATURE-TARGET RELATIONSHIPS
# ─────────────────────────────────────────────────────

features = ["depth", "N_value", "water_content", "void_ratio", "unit_weight"]

fig, axes = plt.subplots(2, 3, figsize=(15, 9))
axes = axes.flatten()

for i, feat in enumerate(features):
    axes[i].scatter(df[feat], df["UCS"], c="steelblue", edgecolors="k",
                    alpha=0.6, s=30)
    axes[i].set_xlabel(feat, fontsize=11)
    axes[i].set_ylabel("UCS (kPa)", fontsize=11)
    axes[i].set_title(f"UCS vs {feat}", fontsize=12)
    axes[i].grid(True, alpha=0.3)

# Correlation values in last subplot
axes[5].axis("off")
corr_text = "Correlation with UCS:\n" + "─" * 25 + "\n"
for feat in features:
    r = df[feat].corr(df["UCS"])
    corr_text += f"{feat:>15s}: {r:+.3f}\n"
axes[5].text(0.1, 0.5, corr_text, fontsize=12, fontfamily="monospace",
             verticalalignment="center", transform=axes[5].transAxes)

plt.suptitle("Feature-Target Relationships", fontsize=14, y=1.01)
plt.tight_layout()
plt.show()

print("N_value and depth show strong positive correlations with UCS.")
print("Water content and void ratio show negative correlations — as expected.")

---

## Part 2: Prepare Data — Feature Matrix and Target Vector

In sklearn, we always separate:
- **X** — the feature matrix (inputs), shape `(n_samples, n_features)`
- **y** — the target vector (output), shape `(n_samples,)`

In [ ]:
# ─────────────────────────────────────────────────────
# SEPARATE FEATURES AND TARGET
# ─────────────────────────────────────────────────────

feature_cols = ["depth", "N_value", "water_content", "void_ratio", "unit_weight"]

X = df[feature_cols].values    # Feature matrix
y = df["UCS"].values           # Target vector

print(f"Feature matrix X: shape {X.shape}")
print(f"Target vector y:  shape {y.shape}")
print()
print(f"Features: {feature_cols}")
print(f"Target:   UCS (kPa)")

---

## Part 3: Train/Test Split

**Why split?** If we train and evaluate on the same data, we cannot tell whether the model truly learned the pattern or just memorized the data.

- **Training set (80%)** — the model learns from this
- **Test set (20%)** — held out for unbiased evaluation

This is the most fundamental concept in ML evaluation.

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN/TEST SPLIT
# ─────────────────────────────────────────────────────

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,       # 20% for testing
    random_state=42      # reproducible split
)

print(f"Training set: {X_train.shape[0]} samples ({X_train.shape[0]/len(X)*100:.0f}%)")
print(f"Test set:     {X_test.shape[0]} samples ({X_test.shape[0]/len(X)*100:.0f}%)")
print()
print(f"X_train shape: {X_train.shape}")
print(f"X_test shape:  {X_test.shape}")
print(f"y_train shape: {y_train.shape}")
print(f"y_test shape:  {y_test.shape}")
print()
print(f"UCS range (train): {y_train.min():.1f} to {y_train.max():.1f} kPa")
print(f"UCS range (test):  {y_test.min():.1f} to {y_test.max():.1f} kPa")

---

## Part 4: Train the Model — `model.fit()`

With sklearn, training a linear regression model takes **exactly 2 lines**:
1. Create the model: `model = LinearRegression()`
2. Fit it to data: `model.fit(X_train, y_train)`

Under the hood, sklearn uses the normal equation (or a numerically stable variant) — exactly what we built from scratch yesterday.

In [ ]:
# ─────────────────────────────────────────────────────
# TRAIN LINEAR REGRESSION
# ─────────────────────────────────────────────────────

model = LinearRegression()
model.fit(X_train, y_train)

print("Model trained!")
print()
print("Learned parameters:")
print(f"  Intercept (b): {model.intercept_:.4f}")
print(f"  Coefficients (w):")
for feat, coef in zip(feature_cols, model.coef_):
    print(f"    {feat:>15s}: {coef:+.4f}")
print()
print("The model equation:")
eq_parts = [f"{coef:+.2f}*{feat}" for feat, coef in zip(feature_cols, model.coef_)]
print(f"  UCS = {model.intercept_:.2f} {' '.join(eq_parts)}")

---

## Part 5: Make Predictions — `model.predict()`

In [ ]:
# ─────────────────────────────────────────────────────
# PREDICTIONS ON TRAIN AND TEST SETS
# ─────────────────────────────────────────────────────

y_train_pred = model.predict(X_train)
y_test_pred = model.predict(X_test)

print("Sample predictions (test set):")
print(f"{'Actual UCS':>12} {'Predicted':>12} {'Error':>10}")
print("─" * 36)
for actual, pred in zip(y_test[:10], y_test_pred[:10]):
    print(f"{actual:12.1f} {pred:12.1f} {actual-pred:+10.1f}")

---

## Part 6: Model Evaluation

In [ ]:
# ─────────────────────────────────────────────────────
# EVALUATION METRICS
# ─────────────────────────────────────────────────────

# Training metrics
r2_train = r2_score(y_train, y_train_pred)
rmse_train = np.sqrt(mean_squared_error(y_train, y_train_pred))
mae_train = mean_absolute_error(y_train, y_train_pred)

# Test metrics
r2_test = r2_score(y_test, y_test_pred)
rmse_test = np.sqrt(mean_squared_error(y_test, y_test_pred))
mae_test = mean_absolute_error(y_test, y_test_pred)

print("Model Evaluation")
print("═" * 50)
print(f"{'Metric':<20} {'Train':>12} {'Test':>12}")
print("─" * 50)
print(f"{'R²':<20} {r2_train:>12.4f} {r2_test:>12.4f}")
print(f"{'RMSE (kPa)':<20} {rmse_train:>12.2f} {rmse_test:>12.2f}")
print(f"{'MAE (kPa)':<20} {mae_train:>12.2f} {mae_test:>12.2f}")
print("═" * 50)
print()
print("Interpretation:")
print(f"  The model explains {r2_test*100:.1f}% of UCS variance on unseen data.")
print(f"  Average prediction error: ±{mae_test:.1f} kPa.")
if abs(r2_train - r2_test) < 0.1:
    print("  Train and test R² are close → no significant overfitting.")
else:
    print("  WARNING: Large gap between train and test R² → possible overfitting.")

In [ ]:
# ─────────────────────────────────────────────────────
# ACTUAL vs PREDICTED PLOT
# ─────────────────────────────────────────────────────

fig, ax = plt.subplots(figsize=(7, 7))

ax.scatter(y_train, y_train_pred, c="steelblue", edgecolors="k",
           alpha=0.6, s=40, label=f"Train (R²={r2_train:.3f})")
ax.scatter(y_test, y_test_pred, c="orange", edgecolors="k",
           alpha=0.8, s=60, label=f"Test (R²={r2_test:.3f})")

# Perfect prediction line
lims = [min(y.min(), y_train_pred.min(), y_test_pred.min()) - 10,
        max(y.max(), y_train_pred.max(), y_test_pred.max()) + 10]
ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect prediction")

ax.set_xlabel("Actual UCS (kPa)", fontsize=12)
ax.set_ylabel("Predicted UCS (kPa)", fontsize=12)
ax.set_title("Actual vs Predicted — Linear Regression", fontsize=14)
ax.legend(fontsize=10)
ax.set_xlim(lims)
ax.set_ylim(lims)
ax.set_aspect("equal")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

print("Points close to the red line = good predictions.")
print("Scatter around the line = prediction uncertainty.")

---

## Part 7: Coefficient Interpretation

The coefficients tell us the **marginal effect** of each feature on UCS, holding all others constant.

In [ ]:
# ─────────────────────────────────────────────────────
# COEFFICIENT IMPORTANCE
# ─────────────────────────────────────────────────────

coef_df = pd.DataFrame({
    "Feature": feature_cols,
    "Coefficient": model.coef_,
    "Abs_Coef": np.abs(model.coef_)
}).sort_values("Abs_Coef", ascending=True)

fig, ax = plt.subplots(figsize=(9, 5))
colors = ["steelblue" if c > 0 else "salmon" for c in coef_df["Coefficient"]]
ax.barh(coef_df["Feature"], coef_df["Coefficient"], color=colors, edgecolor="k")
ax.axvline(x=0, color="black", linewidth=0.8)
ax.set_xlabel("Coefficient Value", fontsize=12)
ax.set_title("Feature Coefficients — Effect on UCS", fontsize=14)
ax.grid(True, alpha=0.3, axis="x")
plt.tight_layout()
plt.show()

print("Coefficient interpretation:")
for _, row in coef_df.iterrows():
    direction = "increases" if row["Coefficient"] > 0 else "decreases"
    print(f"  {row['Feature']:>15s}: +1 unit → UCS {direction} by {abs(row['Coefficient']):.2f} kPa")
print()
print("Note: Coefficients are NOT directly comparable across features")
print("because features have different scales (N ranges 2-60, void ratio 0.3-1.5).")
print("Use standardized coefficients for fair comparison (see exercises).")

---

## Part 8: Residual Analysis

Residuals = Actual - Predicted. A good linear model should have:
1. **Randomly scattered** residuals (no pattern)
2. **Normally distributed** residuals
3. **Constant variance** (homoscedasticity)

In [ ]:
# ─────────────────────────────────────────────────────
# RESIDUAL ANALYSIS
# ─────────────────────────────────────────────────────

residuals_train = y_train - y_train_pred
residuals_test = y_test - y_test_pred

fig, axes = plt.subplots(2, 2, figsize=(13, 10))

# 1. Residuals vs Predicted (test)
axes[0, 0].scatter(y_test_pred, residuals_test, c="orange",
                   edgecolors="k", alpha=0.7, s=50)
axes[0, 0].axhline(y=0, color="red", linestyle="--", linewidth=1.5)
axes[0, 0].set_xlabel("Predicted UCS (kPa)")
axes[0, 0].set_ylabel("Residual (kPa)")
axes[0, 0].set_title("Residuals vs Predicted (Test Set)")
axes[0, 0].grid(True, alpha=0.3)

# 2. Residual histogram
axes[0, 1].hist(residuals_test, bins=8, color="orange", edgecolor="k", alpha=0.7)
axes[0, 1].axvline(x=0, color="red", linestyle="--", linewidth=1.5)
axes[0, 1].set_xlabel("Residual (kPa)")
axes[0, 1].set_ylabel("Frequency")
axes[0, 1].set_title("Residual Distribution (Test Set)")
axes[0, 1].grid(True, alpha=0.3)

# 3. Residuals vs N_value
axes[1, 0].scatter(X_test[:, 1], residuals_test, c="orange",
                   edgecolors="k", alpha=0.7, s=50)
axes[1, 0].axhline(y=0, color="red", linestyle="--", linewidth=1.5)
axes[1, 0].set_xlabel("N_value")
axes[1, 0].set_ylabel("Residual (kPa)")
axes[1, 0].set_title("Residuals vs N_value")
axes[1, 0].grid(True, alpha=0.3)

# 4. Q-Q style: sorted residuals
sorted_res = np.sort(residuals_test)
theoretical_q = np.sort(np.random.normal(0, np.std(residuals_test), len(sorted_res)))
axes[1, 1].scatter(theoretical_q, sorted_res, c="orange",
                   edgecolors="k", alpha=0.7, s=50)
lims_qq = [min(theoretical_q.min(), sorted_res.min()),
           max(theoretical_q.max(), sorted_res.max())]
axes[1, 1].plot(lims_qq, lims_qq, "r--", linewidth=1.5)
axes[1, 1].set_xlabel("Theoretical Quantiles")
axes[1, 1].set_ylabel("Sample Quantiles")
axes[1, 1].set_title("Q-Q Plot (Test Residuals)")
axes[1, 1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

print(f"Residual statistics (test set):")
print(f"  Mean:  {np.mean(residuals_test):.2f} kPa (should be near 0)")
print(f"  Std:   {np.std(residuals_test):.2f} kPa")
print(f"  Min:   {np.min(residuals_test):.2f} kPa")
print(f"  Max:   {np.max(residuals_test):.2f} kPa")

---

## Part 9: Predict for New Soil Samples

In [ ]:
# ─────────────────────────────────────────────────────
# PREDICT UCS FOR NEW SOIL SAMPLES
# ─────────────────────────────────────────────────────

# New samples: [depth, N_value, water_content, void_ratio, unit_weight]
new_samples = np.array([
    [5.0,  10, 32.0, 0.85, 16.5],   # Shallow, weak
    [12.0, 25, 25.0, 0.65, 18.0],   # Medium depth
    [20.0, 45, 18.0, 0.50, 19.5],   # Deep, strong
])

new_predictions = model.predict(new_samples)

print("Predictions for new soil samples:")
print("═" * 65)
print(f"{'Depth':>6} {'N':>4} {'w%':>5} {'e':>5} {'γ':>5} {'UCS_pred':>10}")
print("─" * 65)
for sample, pred in zip(new_samples, new_predictions):
    print(f"{sample[0]:6.1f} {sample[1]:4.0f} {sample[2]:5.1f} {sample[3]:5.2f} {sample[4]:5.1f} {pred:10.1f} kPa")
print("═" * 65)
print()
print("As expected: deeper samples with higher N-values and lower")
print("water content have higher predicted UCS.")

---

## Practice Exercises

### Exercise 1: Standardized Coefficients
Standardize all features to zero mean and unit variance using `StandardScaler`. Retrain the model and compare coefficients — which feature truly has the strongest effect on UCS?

### Exercise 2: Single-Feature Models
Train 5 separate models, each using only one feature. Compare their R² scores. Which single feature best predicts UCS?

### Exercise 3: Different Split Ratios
Try test sizes of 10%, 20%, 30%, and 40%. How does the test R² change? Plot `test_size` vs `R²_test`.

In [ ]:
# EXERCISE 1: Standardized Coefficients
# from sklearn.preprocessing import StandardScaler
# scaler = StandardScaler()
# X_train_scaled = scaler.fit_transform(X_train)
# X_test_scaled = scaler.transform(X_test)
# model_scaled = LinearRegression()
# model_scaled.fit(X_train_scaled, y_train)
# for feat, coef in zip(feature_cols, model_scaled.coef_):
#     print(f"{feat:>15s}: {coef:+.4f}")

In [ ]:
# EXERCISE 2: Single-Feature Models
# for i, feat in enumerate(feature_cols):
#     m = LinearRegression()
#     m.fit(X_train[:, i:i+1], y_train)
#     r2 = r2_score(y_test, m.predict(X_test[:, i:i+1]))
#     print(f"{feat:>15s}: R² = {r2:.4f}")

In [ ]:
# EXERCISE 3: Different Split Ratios
# test_sizes = [0.1, 0.2, 0.3, 0.4]
# r2_scores = []
# for ts in test_sizes:
#     Xtr, Xte, ytr, yte = train_test_split(X, y, test_size=ts, random_state=42)
#     m = LinearRegression().fit(Xtr, ytr)
#     r2_scores.append(r2_score(yte, m.predict(Xte)))
# plt.plot(test_sizes, r2_scores, 'o-')
# plt.xlabel('Test Size'); plt.ylabel('R²')
# plt.show()

---

## Key Takeaways

1. **Scikit-learn** makes linear regression trivial: `model = LinearRegression()`, `model.fit(X, y)`, `model.predict(X_new)`. Three lines replace everything we built from scratch yesterday.

2. **Train/test split** is essential — always evaluate on data the model has never seen. `train_test_split(X, y, test_size=0.2)` is the standard approach.

3. **Multiple features** improve predictions. Our model uses depth, N-value, water content, void ratio, and unit weight together to predict UCS.

4. **Coefficients** tell us the direction and magnitude of each feature's effect. Positive coefficients (N_value, depth) mean higher values increase UCS; negative ones (water_content, void_ratio) decrease it.

5. **Residual analysis** checks model assumptions. Random residuals with constant variance confirm the linear model is appropriate.

---

## How This Connects to ML

| Today's Concept | ML Equivalent |
|----------------|---------------|
| `train_test_split` | Foundation of all ML evaluation |
| `model.fit()` | Training any supervised model |
| `model.predict()` | Inference / deployment |
| R², RMSE, MAE | Standard regression metrics |
| Feature coefficients | Feature importance (linear models) |
| Residual plots | Error analysis for any model |
| Multiple features | Multivariate input to any ML model |
| Actual vs predicted plot | The #1 regression diagnostic plot |

Tomorrow (Day 038) we tackle **polynomial regression and regularization** — what to do when linear isn't enough, and how to prevent overfitting.

---

*Day 037 of 100 — sklearn makes ML accessible, but understanding the math (Day 036) makes you dangerous!*